<a href="https://colab.research.google.com/github/jun-yuaka/deep-learning-llm-notes/blob/main/deep_learning_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import keras
from keras import ops

class NaiveDense:
  def  __init__(self, input_size, output_size, activation=None):
    self.activation = activation
    self.W = keras.Variable(
        shape=(input_size, output_size),
        initializer="uniform"
    )
    self.b = keras.Variable(
        shape=(output_size,),
        initializer="zeros"
    )

  def __call__(self, inputs):
    x = ops.matmul(inputs, self.W)
    x = x + self.b
    if self.activation is not None:
      x = self.activation(x)
    return x

  @property
  def weights(self):
    return [self.W, self.b]

In [5]:
class NaiveSequential:
  def __init__(self, layers):
    self.layers = layers

  def __call__(self, inputs):
    x = inputs
    for layer in self.lyers:
      x = layer(x)
    return x

  @property
  def weights(self):
    weights = []
    for layer in self.layers:
      weights += layer.weights
    return weights

In [6]:
model = NaiveSequential(
    [
        NaiveDense(input_size=28 * 28, output_size=512, activation=ops.relu),
        NaiveDense(input_size=512, output_size=10, activation=ops.softmax),
    ]
)
assert len(model.weights) == 4

In [7]:
import math

class BathGenerator:
  def __init__(self, images, labels, batch_size=128):
    assert len(images) == len(labels)
    self.index = 0
    self.images = images
    self.labels = labels
    self.batch_size = batch_size
    self.num_batches = math.ceil(len(images) / batch_size)

  def next(self):
    images = self.images[self.index : self.index + self.batch_size]
    labels = self.labels[self.index : self.index + self.batch_size]
    self.index += self.batch_size
    return images, labels

In [10]:
def one_training_step(model, images_batch, labels_batch):
  predictions = model(images_batch)
  loss = ops.sparse_categorical_crossentropy(
      labels_batch, predictions
  )
  average_loss = ops.mean(loss)
  gradients = get_gradients_of_loss_wrt_weights(
      loss, model.weights
  )
  update_weights(gradients, model.weights)
  return loss